## Reading the Raw Orders Data from the source

**Purpose:** Loads the raw order-level export from the source system CSV (`/Volumes/workspace/bronze/source_systems/orders_data.csv`) into a Spark DataFrame — reading the header row, inferring column types, and handling quotes and multi-line values for fields with embedded commas or wrapped text.

In [0]:
# Load the data
df = (spark.read.options(
    header='True', 
    inferSchema='True',
    escape='"',
    quote='"',
    multiLine='true'
).csv('/Volumes/workspace/bronze/source_systems/orders_data.csv'))

## Write the Orders Data inside the Bronze Layer

**Purpose:** Promotes the raw orders data to the Bronze layer.

**Steps:**
1. Clean column names — replace spaces with underscores and strip invalid characters (`#`, parentheses, full stops) so every column is a valid table field name.
2. Overwrite `workspace.bronze.orders_data` (1,577 rows), with `overwriteSchema` enabled so re-runs pick up any source schema changes.

In [0]:
# Clean column names by replacing invalid characters with underscores
from pyspark.sql.functions import col

df_clean = df
for column in df.columns:
    clean_name = column.replace(' ', '_').replace('#', 'num').replace('(', '').replace(')', '').replace('.', '')
    df_clean = df_clean.withColumnRenamed(column, clean_name)

df_clean.write.mode('overwrite').option('overwriteSchema', 'true').saveAsTable('workspace.bronze.orders_data')

## Display the data from the Orders Table

**Purpose:** Verifies the newly written bronze orders table by querying it directly from `workspace.bronze.orders_data`.

In [0]:
%sql

SELECT * FROM workspace.bronze.orders_data

## Reading the Raw Categories Data from the source

**Purpose:** Loads the raw category-level sales export from the source system CSV (`/Volumes/workspace/bronze/source_systems/categories.csv`) into a Spark DataFrame, using the same read options as the orders data.

In [0]:
# Load the data
df = (spark.read.options(
    header='True', 
    inferSchema='True',
    escape='"',
    quote='"',
    multiLine='true'
).csv('/Volumes/workspace/bronze/source_systems/categories.csv'))

## Write the Categories Data inside the Bronze Layer

**Purpose:** Promotes the raw categories data to the Bronze layer using the same column-name cleaning, overwriting `workspace.bronze.categories_data` (238 rows).

**Note:** the raw data holds duplicate and near-duplicate category names (e.g. the DTF adhesive powder variants); these are consolidated later in the Silver layer.

In [0]:
# Clean column names by replacing invalid characters with underscores
from pyspark.sql.functions import col

df_clean = df
for column in df.columns:
    clean_name = column.replace(' ', '_').replace('#', 'num').replace('(', '').replace(')', '').replace('.', '')
    df_clean = df_clean.withColumnRenamed(column, clean_name)

df_clean.write.mode('overwrite').option('overwriteSchema', 'true').saveAsTable('workspace.bronze.categories_data')


## Display the Data from the Categories Table

**Purpose:** Reads the bronze categories table back from `workspace.bronze.categories_data` and displays it to verify the ingest.

In [0]:
# Display the data
df_clean = spark.table('workspace.bronze.categories_data')
df_clean.display()


## Reading the Revenue Raw Data from the source

**Purpose:** Loads the raw daily revenue export from the source system CSV (`/Volumes/workspace/bronze/source_systems/revenue.csv`) into a Spark DataFrame, using the same read options as the other sources.

**Note:** `Net_sales` and `Total_sales` land as strings, and a few rows carry a stray apostrophe from CSV escaping; both issues are fixed in the Silver layer.

In [0]:
# Load the data
df = (spark.read.options(
    header='True', 
    inferSchema='True',
    escape='"',
    quote='"',
    multiLine='true'
).csv('/Volumes/workspace/bronze/source_systems/revenue.csv'))

## Write the Revenue Data inside the Bronze Layer

**Purpose:** Promotes the raw revenue data to the Bronze layer using the same column-name cleaning, overwriting `workspace.bronze.revenue_data` (1,097 daily rows).

In [0]:
# Clean column names by replacing invalid characters with underscores
from pyspark.sql.functions import col

df_clean = df
for column in df.columns:
    clean_name = column.replace(' ', '_').replace('#', 'num').replace('(', '').replace(')', '').replace('.', '')
    df_clean = df_clean.withColumnRenamed(column, clean_name)

df_clean.write.mode('overwrite').option('overwriteSchema', 'true').saveAsTable('workspace.bronze.revenue_data')

## Display Data from the Revenue Table

**Purpose:** Reads the bronze revenue table back from `workspace.bronze.revenue_data` and displays it to verify the ingest.

In [0]:
# Display the data
df_clean = spark.table('workspace.bronze.revenue_data')
df_clean.display()